# Lab 14: The 5 Universal API Components

## Naming what you've already been doing

Labs 9-13 all called Groq's chat completions endpoint through `call_groq`. This lab names the shape of that call - endpoint, method, headers, body, response - and wraps it into `SimpleAPIClient`, a small reusable class that isn't specific to Groq or to chat completions at all.

### Setup - get a free key at https://console.groq.com/keys

In [ ]:
"""Get your free Groq API key at https://console.groq.com/keys - it's never written to this notebook."""
import getpass
import json
import requests

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
DEFAULT_MODEL = "llama-3.1-8b-instant"

GROQ_API_KEY = getpass.getpass("Enter your GROQ_API_KEY: ")


def build_request_payload(messages, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The request BODY: what we're asking the model to do."""
    payload = {"model": model, "messages": messages, "temperature": temperature}
    if response_format is not None:
        payload["response_format"] = response_format
    return payload


def extract_reply(response_json):
    """Pulls the assistant's text out of the RESPONSE body."""
    return response_json["choices"][0]["message"]["content"]


def call_groq(messages, api_key=None, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The full request: ENDPOINT + METHOD (POST) + HEADERS (auth) + BODY -> RESPONSE."""
    headers = {
        "Authorization": f"Bearer {api_key or GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = build_request_payload(messages, model=model, temperature=temperature, response_format=response_format)
    response = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return extract_reply(response.json())


print("Groq setup ready. Call call_groq(messages) to send a chat completion request.")


## Exercise

**Goal:** write a small reusable "API client" class. Instead of repeating the same `requests.post(...)` code every time you talk to a web service, the class remembers the service's address and headers once, and you call `.post(path, body)` whenever you need it. The same idea is used later to wrap the Groq API.

## Words you will see

- **API**: a web service you talk to by sending requests over the internet and reading the replies.
- **Endpoint / URL**: the full web address of one API action, for example `https://httpbin.org/post`.
- **Base URL and path**: the base is the shared start of the address (`https://httpbin.org`); the path is the part that picks the action (`/post`). Together they form the full URL.
- **Header**: extra information sent with a request, as key-value pairs, such as `{"X-Test": "1"}` or an API key.
- **JSON**: a text format for data that looks like a Python dict. `requests` can send a dict as JSON (`json=...`) and read a reply back into a dict (`response.json()`).
- **POST request**: a request that sends data to the server (as opposed to a GET that only fetches).
- **Status code**: a number in the reply. `2xx` (like 200) means success; `4xx` and `5xx` mean an error.
- **`raise_for_status()`**: a method on the response that does nothing for 2xx codes and raises `requests.exceptions.HTTPError` otherwise, so errors are not silently ignored.
- **httpbin.org**: a free public "echo" service. Whatever you send to `/post`, it sends back inside its JSON reply, so you can see what you sent.
- **Class, method, attribute**: a class is a blueprint; a method is a function inside it (`client.post(...)`); an attribute is data stored on the object (`self.base_url`).

## What is already written for you

`__init__` is already done. It stores `self.base_url` (with any trailing `/` removed, so `"https://httpbin.org/"` becomes `"https://httpbin.org"`) and `self.headers` (a dict; an empty dict if you pass nothing). You only write `post`.

### `SimpleAPIClient.post(self, path, json_body)`

**Receives:**

- `path` (str), starting with `/`, for example `"/post"`
- `json_body` (dict), the data to send, for example `{"hello": "world"}`

**Must return:** a `dict`, the reply body parsed from JSON.

**The idea in plain words:**

1. Put the base URL and the path together to get the full URL: `"https://httpbin.org"` plus `"/post"` gives `"https://httpbin.org/post"`.
2. Send a POST request to that URL with `requests`, passing `self.headers` as the headers and `json_body` as the JSON body. Passing a timeout (for example `timeout=30`) stops your program from waiting forever if the server never answers.
3. Call `raise_for_status()` on the response, so a 404 or 500 becomes an exception instead of being quietly returned.
4. Return `response.json()`.

**Edge cases the solution handles:** a base URL given with a trailing slash (already handled in `__init__`), a missing `headers` argument (already handled), and a non-2xx reply (raises `HTTPError`, for example `post("/status/404", {})` on httpbin).

**Worked example.** I ran the reference solution against httpbin:

```python
client = SimpleAPIClient("https://httpbin.org", headers={"X-Test": "1"})
reply = client.post("/post", {"hello": "world"})
reply["json"]                  # {'hello': 'world'}   (the body you sent, echoed back)
reply["headers"]["X-Test"]     # '1'                  (your header, echoed back)
```

**Check:** with the example above, `reply["json"] == {"hello": "world"}` is `True` and `type(reply)` is `dict`. If you call `client.post("/status/404", {})` it should raise `requests.exceptions.HTTPError` rather than return.

### What the checks below test

The cell makes **one real call** to httpbin.org, so you need an internet connection. The asserts check that `base_url` is `"https://httpbin.org"`, that the echoed `json` equals `{"hello": "world"}` (so you sent the body correctly), and that the echoed header `X-Test` is `"1"` (so you sent the headers). If you see a connection error, check your internet; httpbin is a public service and is occasionally slow. When every assert passes, the cell prints **All checks passed**, and the message reminds you a real network call was made.

In [ ]:
class SimpleAPIClient:
    def __init__(self, base_url: str, headers: dict | None = None):
        self.base_url = base_url.rstrip("/")
        self.headers = headers or {}

    def post(self, path: str, json_body: dict) -> dict:
        """Send a POST request to base_url + path and return the JSON reply.

        Args:
            path: the part of the address after the base URL, starting with "/"
                (for example "/post").
            json_body: a dict to send as the JSON body (for example {"hello": "world"}).

        Returns:
            dict: the reply body parsed from JSON.

        Raises:
            requests.exceptions.HTTPError: if the server replies with a non-2xx status.

        Example:
            SimpleAPIClient("https://httpbin.org").post("/post", {"hello": "world"})["json"]
            gives {"hello": "world"} because httpbin echoes back what it received.

        TODO:
            1. Build the full URL from self.base_url and path.
            2. Send the POST with `requests`, using self.headers as headers and
               json_body as the JSON body (add a timeout such as 30 seconds).
            3. Call raise_for_status() on the response so errors raise.
            4. Return response.json().
        """
        raise NotImplementedError("SimpleAPIClient.post() is not implemented yet - replace the TODO with your code")


client = SimpleAPIClient(base_url="https://httpbin.org", headers={"X-Test": "1"})
assert client.base_url == "https://httpbin.org", 'client.base_url should be ' + repr("https://httpbin.org") + ", got " + repr(client.base_url)
result = client.post("/post", {"hello": "world"})
assert result["json"] == {"hello": "world"}, 'result["json"] should be ' + repr({"hello": "world"}) + ", got " + repr(result["json"])
assert result["headers"]["X-Test"] == "1", 'result["headers"]["X-Test"] should be ' + repr("1") + ", got " + repr(result["headers"]["X-Test"])

print("All checks passed (this made a real call to httpbin.org, a public echo service).")

## Solution

In [ ]:
class SimpleAPIClient:
    def __init__(self, base_url: str, headers: dict | None = None):
        self.base_url = base_url.rstrip("/")
        self.headers = headers or {}

    def post(self, path: str, json_body: dict) -> dict:
        url = f"{self.base_url}{path}"
        response = requests.post(url, headers=self.headers, json=json_body, timeout=30)
        response.raise_for_status()
        return response.json()


print(SimpleAPIClient("https://httpbin.org").post("/post", {"hello": "world"})["json"])

## Real-world project (LIVE - needs your GROQ_API_KEY above)

Repeats lab 9's zero-shot sentiment call, but through `SimpleAPIClient` instead of the raw `requests.post` in `call_groq` - same 5 components, same result, different wrapping. Reuses `build_zero_shot_messages` / `parse_label` from lab 9 (redefined here so this notebook stays self-contained).

In [ ]:
def build_zero_shot_messages(review: str) -> list[dict]:
    return [
        {
            "role": "system",
            "content": "Classify the sentiment of the review as exactly one word: "
                       "'positive' or 'negative'. Reply with nothing else.",
        },
        {"role": "user", "content": review},
    ]


def parse_label(reply_text: str) -> str:
    normalized = reply_text.strip().strip(".").lower()
    if "negative" in normalized:
        return "negative"
    if "positive" in normalized:
        return "positive"
    return normalized


groq_client = SimpleAPIClient(
    base_url="https://api.groq.com/openai/v1",
    headers={"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"},
)

review = "The battery life on this thing is incredible."
response_body = groq_client.post(
    "/chat/completions",
    {"model": DEFAULT_MODEL, "messages": build_zero_shot_messages(review), "temperature": 0.0},
)
reply_text = response_body["choices"][0]["message"]["content"]
print(f"Review: {review}")
print(f"Predicted sentiment: {parse_label(reply_text)}")